# Format Evaluation — Local Models

Score three local Gemma 4 variants on reasoning traces captured by the Knowable macOS app **in the last hour**.

**Models compared**
- `gemma4:e4b` — baseline (unmodified Gemma 4 E4B)
- `gemma4:knowable-tuned-hf` — PEFT/TRL LoRA fine-tune (merged + quantized)
- `gemma4:knowable-tuned-unsloth` — Unsloth LoRA fine-tune (merged + quantized)

**Criteria (three only, per spec)**
1. Output parses as JSON
2. All five required keys present: `understanding`, `events`, `hint`, `hint_speech`, `state`
3. `events` is a valid array

The eval re-runs each captured trace's input through each model under the same conditions Sonnet saw (same system prompt, same multimodal user message). Each model runs to completion against all traces, then is evicted from VRAM via `keep_alive=0` before the next model loads — required because we have 24 GB of VRAM and each model is ~9.6 GB.

**Run cells top to bottom.** Expect ~10–15 min wall-clock for 40 traces × 3 models on a single-GPU machine.


In [1]:
from __future__ import annotations

import base64
import json
import os
import subprocess
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import requests

# ---- Config ----
BUCKET = "knowable-finetune-traces"
AWS_PROFILE = os.environ.get("AWS_PROFILE", "knowable")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
OLLAMA_URL = "http://localhost:11434"

MODELS: list[str] = [
    "gemma4:e4b",
    "gemma4:knowable-tuned-hf",
    "gemma4:knowable-tuned-unsloth",
]

# Look-back window for "recent" traces (uses S3 LastModified).
LOOKBACK_HOURS = 1

# Per-trace inference timeout. Generous because the first request after a
# model swap triggers a cold load (~30–60 s on M-series), then steady ~3 s.
INFERENCE_TIMEOUT_S = 180

TRACES_DIR = Path("./format_eval_traces")
OUTPUTS_DIR = Path("./format_eval_outputs")
TRACES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# JSON Schema mirrors the production Swift client at
# Knowable/Services/Reasoning/LocalReasoningBackend.swift so the eval
# runs under the SAME format constraint the app applies in production.
JSON_SCHEMA: dict[str, Any] = {
    "type": "object",
    "required": ["understanding", "events", "hint", "hint_speech", "state"],
    "additionalProperties": False,
    "properties": {
        "understanding": {"type": "string", "maxLength": 2000},
        "events": {
            "type": "array",
            "maxItems": 20,
            "items": {"type": "string"},
        },
        "hint": {"type": ["string", "null"], "maxLength": 500},
        "hint_speech": {"type": ["string", "null"], "maxLength": 500},
        "state": {
            "type": "string",
            "enum": ["active", "camera_lost", "positioning_camera"],
        },
    },
}

print(f"Models to evaluate: {MODELS}")
print(f"Lookback window:    {LOOKBACK_HOURS} h")


Models to evaluate: ['gemma4:e4b', 'gemma4:knowable-tuned-hf', 'gemma4:knowable-tuned-unsloth']
Lookback window:    1 h


## Sync traces from S3

Lists manifests in the bucket, filters to those whose S3 `LastModified` falls within the look-back window, downloads the manifest + frame images for each. Idempotent — re-running skips files already on disk.

The "now" anchor is the timestamp of the most recent manifest in the bucket, not the local wall clock. This lets the notebook re-run identically against a frozen dataset even after time has passed.


In [17]:
result = subprocess.run(
    ["aws", "s3", "ls", f"s3://{BUCKET}/traces/", "--recursive",
     "--profile", AWS_PROFILE, "--region", AWS_REGION],
    capture_output=True, text=True, check=True,
)
manifest_lines = [l for l in result.stdout.splitlines() if l.strip().endswith("manifest.json")]


def parse_ts(line: str) -> datetime:
    return datetime.strptime(" ".join(line.split()[:2]), "%Y-%m-%d %H:%M:%S")


entries = sorted([(parse_ts(l), l.split()[-1]) for l in manifest_lines], reverse=True)
if not entries:
    raise RuntimeError("No manifests in bucket — enable trace capture in the app first.")

newest = datetime(2026, 5, 14, 21, 42, 24)
cutoff = newest - timedelta(hours=LOOKBACK_HOURS)
recent = [(ts, key) for ts, key in entries if ts >= cutoff and ts <= newest]
print(f"Newest manifest (frozen in time):    {newest}")
print(f"Cutoff (lookback):  {cutoff}")
print(f"Traces in window:   {len(recent)}")


def fetch_trace(s3_key: str) -> Path:
    parts = s3_key.split("/")  # e.g. ["traces", "2026-05-15", "<uuid>", "manifest.json"]
    local_dir = TRACES_DIR / "/".join(parts[1:-1])
    local_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = local_dir / "manifest.json"
    if not manifest_path.exists():
        subprocess.run(
            ["aws", "s3", "cp", f"s3://{BUCKET}/{s3_key}", str(manifest_path),
             "--profile", AWS_PROFILE, "--region", AWS_REGION, "--quiet"],
            check=True,
        )
    manifest = json.loads(manifest_path.read_text())
    for frame_name in manifest["request"]["frame_files"]:
        frame_local = local_dir / frame_name
        if not frame_local.exists():
            frame_s3 = f"traces/{'/'.join(parts[1:-1])}/{frame_name}"
            subprocess.run(
                ["aws", "s3", "cp", f"s3://{BUCKET}/{frame_s3}", str(frame_local),
                 "--profile", AWS_PROFILE, "--region", AWS_REGION, "--quiet"],
                check=True,
            )
    return manifest_path


manifest_paths = [fetch_trace(k) for _, k in recent]
print(f"Fetched {len(manifest_paths)} manifests + their frames → {TRACES_DIR}/")


Newest manifest (frozen in time):    2026-05-14 21:42:24
Cutoff (lookback):  2026-05-14 20:42:24
Traces in window:   40
Fetched 40 manifests + their frames → format_eval_traces/


## Helpers — inference, format check, VRAM eviction

These wrap the Ollama HTTP API and the three formatting criteria. All cells below this one call into these helpers.


In [3]:
def format_user_text(manifest: dict) -> str:
    """Render request fields as the same text block the training pipeline uses."""
    req = manifest["request"]
    flags = req["flags"]
    uq = flags.get("user_query")
    uq_line = f"user_query: {uq}" if uq else "user_query: (none)"
    return (
        f"FLAGS:\n"
        f"  is_milo_speaking: {flags['is_milo_speaking']}\n"
        f"  force_reply: {flags['force_reply']}\n"
        f"  {uq_line}\n"
        f"\n"
        f"CURRENT_ANALYSIS:\n{req['current_analysis'] or '(empty)'}\n"
        f"\n"
        f"EVENT_LOG:\n{req['event_log'] or '(empty)'}\n"
    )


def load_frames_b64(manifest_path: Path, manifest: dict) -> list[str]:
    frames = []
    for name in manifest["request"]["frame_files"]:
        path = manifest_path.parent / name
        frames.append(base64.b64encode(path.read_bytes()).decode())
    return frames


def run_inference(model: str, manifest_path: Path) -> dict:
    """Single Ollama chat call. Returns {trace_id, raw_text}."""
    manifest = json.loads(manifest_path.read_text())
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": manifest["system_prompt"]},
            {
                "role": "user",
                "content": format_user_text(manifest),
                "images": load_frames_b64(manifest_path, manifest),
            },
        ],
        "stream": False,
        "format": JSON_SCHEMA,
        "options": {"temperature": 0.4},
    }
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=INFERENCE_TIMEOUT_S)
    r.raise_for_status()
    return {
        "trace_id": manifest["trace_id"],
        "raw_text": r.json()["message"]["content"],
    }


def check_format(raw_text: str) -> dict:
    """Three criteria, per the eval spec."""
    try:
        obj = json.loads(raw_text)
    except json.JSONDecodeError as e:
        return {
            "valid_json": False,
            "all_keys_present": False,
            "events_is_array": False,
            "missing_keys": [],
            "json_error": str(e)[:160],
        }
    required = {"understanding", "events", "hint", "hint_speech", "state"}
    present = set(obj.keys())
    return {
        "valid_json": True,
        "all_keys_present": required.issubset(present),
        "events_is_array": isinstance(obj.get("events"), list),
        "missing_keys": sorted(required - present),
        "json_error": None,
    }


def unload(model: str) -> None:
    """Evict the model from VRAM. Idempotent; swallows network errors."""
    try:
        requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={"model": model, "keep_alive": 0},
            timeout=10,
        )
    except requests.exceptions.RequestException:
        pass


def evaluate_model(model: str, manifest_paths: list[Path]) -> list[dict]:
    print(f"\n=== {model} ===")
    safe_name = model.replace(':', '_').replace('/', '_')
    out_file = OUTPUTS_DIR / f"{safe_name}.jsonl"
    results = []
    t0 = time.time()
    for i, mp in enumerate(manifest_paths, 1):
        try:
            r = run_inference(model, mp)
            r.update(check_format(r["raw_text"]))
        except Exception as e:
            r = {
                "trace_id": mp.parent.name,
                "raw_text": "",
                "valid_json": False,
                "all_keys_present": False,
                "events_is_array": False,
                "missing_keys": [],
                "json_error": f"inference exception: {type(e).__name__}: {e}",
            }
        results.append(r)
        elapsed = time.time() - t0
        rate = i / max(elapsed, 1e-3)
        eta_s = (len(manifest_paths) - i) / max(rate, 1e-3)
        flag = "OK " if (r["valid_json"] and r["all_keys_present"] and r["events_is_array"]) else "FAIL"
        print(
            f"  [{i:2d}/{len(manifest_paths)}] {r['trace_id'][:8]}…  "
            f"{flag}  json={r['valid_json']} keys={r['all_keys_present']} events={r['events_is_array']}  "
            f"ETA {eta_s:.0f}s",
            flush=True,
        )
    with open(out_file, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
    print(f"Wrote {len(results)} results → {out_file}")
    return results


## Run — `gemma4:e4b` (baseline)

Loads the base Gemma 4 E4B model and runs all traces. First call triggers a cold load (~30–60 s); subsequent calls run at steady state.


In [4]:
results_base = evaluate_model(MODELS[0], manifest_paths)
unload(MODELS[0])



=== gemma4:e4b ===
  [ 1/40] efa31927…  FAIL  json=False keys=False events=False  ETA 717s
  [ 2/40] fcc3371a…  FAIL  json=False keys=False events=False  ETA 791s
  [ 3/40] adfb729f…  FAIL  json=False keys=False events=False  ETA 661s
  [ 4/40] 5978f283…  OK   json=True keys=True events=True  ETA 810s
  [ 5/40] e6d2c13f…  FAIL  json=False keys=False events=False  ETA 724s
  [ 6/40] e8bd9b32…  FAIL  json=False keys=False events=False  ETA 649s
  [ 7/40] 6783b6f8…  OK   json=True keys=True events=True  ETA 654s
  [ 8/40] cf6f2581…  OK   json=True keys=True events=True  ETA 638s
  [ 9/40] 974fd9e0…  OK   json=True keys=True events=True  ETA 574s
  [10/40] 94c23d72…  OK   json=True keys=True events=True  ETA 513s
  [11/40] cf0911cb…  FAIL  json=False keys=False events=False  ETA 473s
  [12/40] ee716021…  OK   json=True keys=True events=True  ETA 465s
  [13/40] 2f141d9d…  FAIL  json=False keys=False events=False  ETA 441s
  [14/40] 1c8222ec…  FAIL  json=False keys=False events=False  ETA 4

## Run — `gemma4:knowable-tuned-hf` (PEFT LoRA)


In [5]:
results_peft = evaluate_model(MODELS[1], manifest_paths)
unload(MODELS[1])



=== gemma4:knowable-tuned-hf ===
  [ 1/40] efa31927…  OK   json=True keys=True events=True  ETA 990s
  [ 2/40] fcc3371a…  OK   json=True keys=True events=True  ETA 802s
  [ 3/40] adfb729f…  OK   json=True keys=True events=True  ETA 916s
  [ 4/40] 5978f283…  OK   json=True keys=True events=True  ETA 860s
  [ 5/40] e6d2c13f…  OK   json=True keys=True events=True  ETA 831s
  [ 6/40] e8bd9b32…  OK   json=True keys=True events=True  ETA 776s
  [ 7/40] 6783b6f8…  OK   json=True keys=True events=True  ETA 759s
  [ 8/40] cf6f2581…  OK   json=True keys=True events=True  ETA 718s
  [ 9/40] 974fd9e0…  OK   json=True keys=True events=True  ETA 657s
  [10/40] 94c23d72…  OK   json=True keys=True events=True  ETA 707s
  [11/40] cf0911cb…  OK   json=True keys=True events=True  ETA 657s
  [12/40] ee716021…  OK   json=True keys=True events=True  ETA 608s
  [13/40] 2f141d9d…  OK   json=True keys=True events=True  ETA 585s
  [14/40] 1c8222ec…  OK   json=True keys=True events=True  ETA 570s
  [15/40] cbfa

## Run — `gemma4:knowable-tuned-unsloth` (Unsloth LoRA)


In [6]:
results_unsloth = evaluate_model(MODELS[2], manifest_paths)
unload(MODELS[2])



=== gemma4:knowable-tuned-unsloth ===
  [ 1/40] efa31927…  FAIL  json=False keys=False events=False  ETA 6082s
  [ 2/40] fcc3371a…  OK   json=True keys=True events=True  ETA 3877s
  [ 3/40] adfb729f…  OK   json=True keys=True events=True  ETA 2666s
  [ 4/40] 5978f283…  OK   json=True keys=True events=True  ETA 2058s
  [ 5/40] e6d2c13f…  OK   json=True keys=True events=True  ETA 1814s
  [ 6/40] e8bd9b32…  OK   json=True keys=True events=True  ETA 1551s
  [ 7/40] 6783b6f8…  OK   json=True keys=True events=True  ETA 1446s
  [ 8/40] cf6f2581…  OK   json=True keys=True events=True  ETA 1327s
  [ 9/40] 974fd9e0…  OK   json=True keys=True events=True  ETA 1200s
  [10/40] 94c23d72…  OK   json=True keys=True events=True  ETA 1073s
  [11/40] cf0911cb…  OK   json=True keys=True events=True  ETA 975s
  [12/40] ee716021…  OK   json=True keys=True events=True  ETA 883s
  [13/40] 2f141d9d…  OK   json=True keys=True events=True  ETA 817s
  [14/40] 1c8222ec…  OK   json=True keys=True events=True  ETA 

## Summary table

Pass rate (%) on each criterion, across all traces, for each model. Higher is better; green = pass, red = fail.


In [7]:
import pandas as pd


def summarize(name: str, results: list[dict]) -> dict:
    n = len(results)
    return {
        "model": name,
        "n": n,
        "valid_json_pct": 100.0 * sum(r["valid_json"] for r in results) / max(n, 1),
        "all_keys_pct": 100.0 * sum(r["all_keys_present"] for r in results) / max(n, 1),
        "events_array_pct": 100.0 * sum(r["events_is_array"] for r in results) / max(n, 1),
        "all_three_pct": 100.0 * sum(
            r["valid_json"] and r["all_keys_present"] and r["events_is_array"]
            for r in results
        ) / max(n, 1),
    }


summary = pd.DataFrame([
    summarize(MODELS[0], results_base),
    summarize(MODELS[1], results_peft),
    summarize(MODELS[2], results_unsloth),
])

styled = summary.style.format({
    "valid_json_pct": "{:.1f}%",
    "all_keys_pct": "{:.1f}%",
    "events_array_pct": "{:.1f}%",
    "all_three_pct": "{:.1f}%",
}).background_gradient(
    subset=["valid_json_pct", "all_keys_pct", "events_array_pct", "all_three_pct"],
    cmap="RdYlGn", vmin=0, vmax=100,
)
styled


,model,n,valid_json_pct,all_keys_pct,events_array_pct,all_three_pct
0,gemma4:e4b,40,65.0%,65.0%,65.0%,65.0%
1,gemma4:knowable-tuned-hf,40,100.0%,100.0%,100.0%,100.0%
2,gemma4:knowable-tuned-unsloth,40,97.5%,97.5%,97.5%,97.5%


## Sample failures per model


In [8]:
def show_failures(name: str, results: list[dict], n: int = 3):
    failures = [
        r for r in results
        if not (r["valid_json"] and r["all_keys_present"] and r["events_is_array"])
    ]
    print(f"\n{name}: {len(failures)} / {len(results)} formatting failures")
    for r in failures[:n]:
        print(f"\n--- {r['trace_id'][:12]}… ---")
        print(
            f"  valid_json={r['valid_json']}  "
            f"all_keys={r['all_keys_present']}  "
            f"events_array={r['events_is_array']}"
        )
        if r.get("json_error"):
            print(f"  json_error: {r['json_error']}")
        if r.get("missing_keys"):
            print(f"  missing_keys: {r['missing_keys']}")
        raw = (r.get("raw_text") or "")[:400]
        print(f"  raw[:400]: {raw!r}")


show_failures(MODELS[0], results_base)
show_failures(MODELS[1], results_peft)
show_failures(MODELS[2], results_unsloth)



gemma4:e4b: 14 / 40 formatting failures

--- efa31927-dd2… ---
  valid_json=False  all_keys=False  events_array=False
  json_error: Expecting value: line 1 column 1 (char 0)
  raw[:400]: 'UNDERSTOOD.\nThe user is currently stuck on the step of finding two numbers that multiply to -12 and add up to 5.\nThe goal is to guide the user to test pairs of factors systematically.\n\nPlan:\n1. Acknowledge the current step (finding two numbers).\n2. Guide the user to list the factors of 12.\n3. Systematically test the pairs to find the correct combination (which is 8 and -3).\n\nExecuting the plan.\n(S'

--- fcc3371a-bbc… ---
  valid_json=False  all_keys=False  events_array=False
  json_error: Expecting value: line 1 column 1 (char 0)
  raw[:400]: "UNDERSTOOD. The student has correctly identified the next step: finding two numbers that multiply to -12 and add up to 5.\n\nThe pairs are:\n1. 1 and -12 (Sum: -11)\n2. -1 and 12 (Sum: 11)\n3. 2 and -6 (Sum: -4)\n4. -2 and 6 (Sum: 4)\n5. 3 and -4 (Sum